# Baseline: nightly resting-heart-rate anomaly detection on real data

NASA Space Apps 2026 · *Create Health Monitoring Software for Astronauts on Space Missions*

This notebook compares each night's sleeping heart rate and sleep with **the same person's other nights**, and flags nights that are unusual for them. It also builds a lab panel from **real NASA astronaut bloodwork**.

| Source | Label in the app | Used for |
| --- | --- | --- |
| PhysioNet multi-night HR + sleep (47 healthy adults on **Earth**, 253 nights) | General human data (PhysioNet) | Nightly model |
| NASA OSDR Inspiration4, OSD-569 (CBC) + OSD-575 (CMP), 4 civilian crew | NASA Inspiration4 | Lab-baseline panel |

The two sources are never merged. The output is an anomaly score with a reason, not a diagnosis.

**Before running:** upload `bidsleep_nights.csv` and `i4_labs_with_baseline.csv` when the first cell asks, then **Runtime → Run all**.

In [ ]:
import os, json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupKFold

try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 160)

NEEDED = ["bidsleep_nights.csv", "i4_labs_with_baseline.csv"]
missing = [f for f in NEEDED if not Path(f).exists()]
if missing:
    try:
        from google.colab import files
        print("Upload:", ", ".join(missing))
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put these next to the notebook: {missing}")
print("Files ready:", [f for f in NEEDED if Path(f).exists()])

## Optional: rebuild the nightly table from raw PhysioNet files

Skip this cell if you uploaded `bidsleep_nights.csv`. It only runs when a `bidsleep/` folder of raw files is present: one folder per person, holding night folders with `hr.csv` and `labels.mat`.

It uses the **expert** sleep-stage labels and reads the start time as US Eastern time. Heart-rate readings outside the labelled recording are dropped, and duplicate timestamps are removed.

In [ ]:
EPOCH_S = 30
SLEEP_STAGES = {1, 2, 3, 4}


def night_features(night_dir: Path):
    m = sio.loadmat(night_dir / "labels.mat")
    stages = np.ravel(m["expert_label"]).astype(int)
    rec_start = str(np.ravel(m["recStart"])[0])
    t0 = datetime.strptime(rec_start, "%Y-%m-%d %H:%M:%S").replace(
        tzinfo=ZoneInfo("America/New_York")).timestamp()

    hr = pd.read_csv(night_dir / "hr.csv", header=None, names=["t", "hr"])
    malformed = int(hr.isna().any(axis=1).sum())
    hr = hr.dropna()
    duplicates = int(hr.duplicated(subset="t").sum())
    hr = hr.drop_duplicates(subset="t")

    # Which 30 s epoch each HR reading falls in; readings outside the labelled
    # recording are dropped.
    epoch = np.floor((hr["t"].to_numpy() - t0) / EPOCH_S).astype(int)
    inside = (epoch >= 0) & (epoch < len(stages))
    dropped = int((~inside).sum())
    hr, epoch = hr[inside], epoch[inside]
    asleep = np.isin(stages[epoch], list(SLEEP_STAGES))
    sleep_hr = hr["hr"].to_numpy()[asleep]

    n = len(stages)
    counts = np.bincount(stages, minlength=6)
    sleep_epochs = counts[1:5].sum()
    total_sleep_h = sleep_epochs * EPOCH_S / 3600
    return {
        "subject_id": night_dir.parent.name,
        "night_folder": night_dir.name,
        "rec_start_et": rec_start,
        "recorded_h": round(n * EPOCH_S / 3600, 2),
        "total_sleep_h": round(total_sleep_h, 2),
        "wake_pct": round(counts[0] / n * 100, 1),
        "n3_pct_of_sleep": round(counts[3] / max(sleep_epochs, 1) * 100, 1),
        "rem_pct_of_sleep": round(counts[4] / max(sleep_epochs, 1) * 100, 1),
        "unknown_pct": round(counts[5] / n * 100, 1),
        "sleep_hr_mean": round(float(sleep_hr.mean()), 2) if len(sleep_hr) else np.nan,
        "sleep_hr_median": float(np.median(sleep_hr)) if len(sleep_hr) else np.nan,
        "sleep_hr_p5": float(np.percentile(sleep_hr, 5)) if len(sleep_hr) else np.nan,
        "sleep_hr_std": round(float(sleep_hr.std(ddof=1)), 2) if len(sleep_hr) > 1 else np.nan,
        "hr_readings_asleep": int(len(sleep_hr)),
        "hr_per_sleep_hour": round(len(sleep_hr) / total_sleep_h, 1) if total_sleep_h else np.nan,
        "hr_dropped_outside_labels": dropped,
        "hr_duplicate_timestamps": duplicates,
        "hr_malformed_rows": malformed,
    }


def main(raw: str, out: str):
    nights = sorted(p.parent for p in Path(raw).glob("*/*/labels.mat"))
    df = pd.DataFrame(night_features(p) for p in nights)
    df = df.sort_values(["subject_id", "rec_start_et"]).reset_index(drop=True)
    df["night_index"] = df.groupby("subject_id").cumcount() + 1
    df["source"] = "PhysioNet bidsleep v1.0.1 (general human data, Earth)"
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    df.to_csv(out, index=False)
    print(f"{len(df)} nights from {df['subject_id'].nunique()} people -> {out}")


if Path("bidsleep").exists():
    import scipy.io as sio
    main("bidsleep", "bidsleep_nights.csv")
else:
    print("No raw bidsleep/ folder: using the uploaded bidsleep_nights.csv")

## 1. Load the nights and flag data-quality problems

Only data problems are flagged: low heart-rate coverage, a doubled reading rate, or more than 15% unknown sleep stage. These nights never count towards anyone's baseline.

A genuinely bad night, such as one with 42% of the time awake, is **not** a data problem. That's what the detector should find.

In [ ]:
# Feature -> which direction is bad
FEATURES = {
    "sleep_hr_median": "high",
    "sleep_hr_p5": "high",       # the resting floor
    "total_sleep_h": "low",
    "wake_pct": "high",
    "n3_pct_of_sleep": "low",
    "rem_pct_of_sleep": "low",
}
LABELS = {
    "sleep_hr_median": ("Sleeping heart rate", "bpm", 0),
    "sleep_hr_p5": ("Lowest sleeping heart rate (5th pct)", "bpm", 0),
    "total_sleep_h": ("Total sleep", "h", 1),
    "wake_pct": ("Time awake", "%", 0),
    "n3_pct_of_sleep": ("Deep sleep (N3)", "% of sleep", 0),
    "rem_pct_of_sleep": ("REM sleep", "% of sleep", 0),
}
HR_FEATURES = ["sleep_hr_median", "sleep_hr_p5"]
SLEEP_FEATURES = ["total_sleep_h", "wake_pct", "n3_pct_of_sleep", "rem_pct_of_sleep"]
TARGET_FALSE_ALARM = 0.05
SHIFTS_BPM = [3, 5, 8, 12]


def add_quality_flags(df):
    """Data-quality problems only. A genuinely bad night (lots of wake) is NOT a flag."""
    df = df.copy()
    reasons = pd.Series("", index=df.index)
    reasons[df["hr_per_sleep_hour"] < 500] += "low HR coverage; "
    reasons[df["hr_per_sleep_hour"] > 1000] += "about double the normal HR rate; "
    reasons[df["unknown_pct"] > 15] += "over 15% unknown sleep stage; "
    df["quality_issue"] = reasons.str.rstrip("; ")
    df["clean"] = df["quality_issue"] == ""
    return df


def loo_deviations(df):
    """Each night minus the median of the same person's other clean nights."""
    out = []
    for _, g in df.groupby("subject_id"):
        clean = g[g["clean"]]
        for idx, row in g.iterrows():
            others = clean.drop(index=idx, errors="ignore")
            dev = {f: row[f] - others[f].median() if len(others) >= 2 else np.nan for f in FEATURES}
            out.append({"idx": idx, "n_baseline": len(others), **{f"dev_{f}": v for f, v in dev.items()}})
    return pd.DataFrame(out).set_index("idx")


def pooled_spread(dev, people):
    """Robust spread (1.4826 x MAD) of leave-one-out deviations among the given people."""
    sub = dev[dev["subject_id"].isin(people) & dev["clean"]]
    return {f: 1.4826 * np.nanmedian(np.abs(sub[f"dev_{f}"] - np.nanmedian(sub[f"dev_{f}"])))
            for f in FEATURES}


def z_scores(dev, spread):
    z = pd.DataFrame(index=dev.index)
    for f, direction in FEATURES.items():
        raw = dev[f"dev_{f}"] / spread[f]
        z[f] = raw if direction == "high" else -raw  # positive = worse
    return z


def plant(df, rng):
    """Copies of clean nights with planted changes. Baselines still use the originals."""
    base = df[df["clean"]]
    rows = []
    for idx, r in base.iterrows():
        for s in SHIFTS_BPM:
            p = r.copy()
            p["sleep_hr_median"] += s
            p["sleep_hr_p5"] += s
            rows.append({**p, "orig_idx": idx, "plant": f"HR +{s} bpm"})
        p = r.copy()  # disrupted night: less sleep, more wake, less deep sleep
        lost = min(1.5, p["total_sleep_h"] - 2)
        p["total_sleep_h"] -= lost
        p["wake_pct"] += 15
        p["n3_pct_of_sleep"] = max(0, p["n3_pct_of_sleep"] - 8)
        rows.append({**p, "orig_idx": idx, "plant": "Disrupted sleep"})
    return pd.DataFrame(rows)


def deviations_for_planted(df, planted):
    """Deviation of each planted copy from the person's other clean ORIGINAL nights."""
    out = []
    for _, r in planted.iterrows():
        others = df[(df["subject_id"] == r["subject_id"]) & df["clean"]].drop(index=r["orig_idx"])
        out.append({f"dev_{f}": r[f] - others[f].median() if len(others) >= 2 else np.nan for f in FEATURES})
    return pd.DataFrame(out, index=planted.index)


def _fmt(f, v, diff=False):
    name, unit, digits = LABELS[f]
    if unit.startswith("%"):
        return f"{v:.0f} points" if diff else f"{v:.0f}%"
    return f"{v:.{digits}f} {unit}"


def explain(zrow, raw_row, dev_row, top=2):
    """Plain-English reasons for the worst features of one night."""
    parts = []
    for f in zrow.sort_values(ascending=False).index[:top]:
        if not zrow[f] >= 1.5:
            continue
        d = dev_row[f"dev_{f}"]
        parts.append(f"{LABELS[f][0]} {_fmt(f, raw_row[f])} is {_fmt(f, abs(d), diff=True)} "
                     f"{'above' if d > 0 else 'below'} your usual ({zrow[f]:.1f}x a typical night-to-night change)")
    return "; ".join(parts) if parts else "Within your usual range"


def evaluate(nights, seed=0, n_splits=5):
    """Leave-people-out evaluation of the z-score and Isolation Forest detectors."""
    rng = np.random.default_rng(seed)
    df = add_quality_flags(nights)
    dev = loo_deviations(df).join(df[["subject_id", "clean"]])
    ok = dev[[f"dev_{f}" for f in FEATURES]].notna().all(axis=1)
    planted = plant(df, rng)
    pdev = deviations_for_planted(df, planted)
    pdev["subject_id"] = planted["subject_id"].to_numpy()
    pdev["plant"] = planted["plant"].to_numpy()
    pdev = pdev[pdev[[f"dev_{f}" for f in FEATURES]].notna().all(axis=1)]

    people = np.array(sorted(df["subject_id"].unique()))
    results = []
    for train_idx, test_idx in GroupKFold(n_splits=n_splits).split(people, groups=people):
        train_p, test_p = people[train_idx], people[test_idx]
        spread = pooled_spread(dev, train_p)  # never uses the people being scored

        z_tr = z_scores(dev[ok & dev["subject_id"].isin(train_p) & dev["clean"]], spread)
        z_te = z_scores(dev[ok & dev["subject_id"].isin(test_p) & dev["clean"]], spread)
        z_pl = z_scores(pdev[pdev["subject_id"].isin(test_p)], spread)
        plant_lbl = pdev.loc[z_pl.index, "plant"]

        # Detector 1: z-score. HR plants are judged on HR features, sleep plants on sleep features.
        for name, feats in [("all", list(FEATURES)), ("hr", HR_FEATURES), ("sleep", SLEEP_FEATURES)]:
            thr = np.quantile(z_tr[feats].max(axis=1), 1 - TARGET_FALSE_ALARM)
            results.append({"method": "z-score", "group": name, "kind": "untouched",
                            "hits": int((z_te[feats].max(axis=1) > thr).sum()), "n": len(z_te)})
            for lbl in plant_lbl.unique():
                m = plant_lbl == lbl
                results.append({"method": "z-score", "group": name, "kind": lbl,
                                "hits": int((z_pl[m][feats].max(axis=1) > thr).sum()), "n": int(m.sum())})

        # Detector 1b: two separate checks (heart rate, sleep), each with half the false-alarm budget.
        half = TARGET_FALSE_ALARM / 2
        t_hr = np.quantile(z_tr[HR_FEATURES].max(axis=1), 1 - half)
        t_sl = np.quantile(z_tr[SLEEP_FEATURES].max(axis=1), 1 - half)
        flag = lambda z: (z[HR_FEATURES].max(axis=1) > t_hr) | (z[SLEEP_FEATURES].max(axis=1) > t_sl)
        results.append({"method": "z-score", "group": "two", "kind": "untouched", "hits": int(flag(z_te).sum()), "n": len(z_te)})
        for lbl in plant_lbl.unique():
            m = (plant_lbl == lbl).to_numpy()
            results.append({"method": "z-score", "group": "two", "kind": lbl, "hits": int(flag(z_pl[m]).sum()), "n": int(m.sum())})

        # Detector 2: Isolation Forest on all six personal z-features.
        model = IsolationForest(n_estimators=300, random_state=seed).fit(z_tr)
        thr = np.quantile(-model.score_samples(z_tr), 1 - TARGET_FALSE_ALARM)
        results.append({"method": "Isolation Forest", "group": "if", "kind": "untouched",
                        "hits": int((-model.score_samples(z_te) > thr).sum()), "n": len(z_te)})
        for lbl in plant_lbl.unique():
            m = (plant_lbl == lbl).to_numpy()
            results.append({"method": "Isolation Forest", "group": "if", "kind": lbl,
                            "hits": int((-model.score_samples(z_pl[m]) > thr).sum()), "n": int(m.sum())})
    return pd.DataFrame(results)


def summary_table(res):
    agg = res.groupby(["method", "group", "kind"])[["hits", "n"]].sum()
    agg["rate"] = agg["hits"] / agg["n"]
    rows = []
    for (method, group), g in agg.groupby(level=[0, 1]):
        g = g.droplevel([0, 1])
        label = {"all": "z-score, one combined score", "two": "z-score, separate HR and sleep checks (app)",
                 "hr": "z-score, heart rate only",
                 "sleep": "z-score, sleep only", "if": "Isolation Forest, all 6 features"}[group]
        row = {"Detector": label, "False alarms (untouched nights)": f"{g.loc['untouched', 'rate']:.0%}"}
        for k in [f"HR +{s} bpm" for s in SHIFTS_BPM] + ["Disrupted sleep"]:
            row[k] = f"{g.loc[k, 'rate']:.0%}"
        rows.append(row)
    order = ["z-score, separate HR and sleep checks (app)", "z-score, one combined score", "Isolation Forest, all 6 features",
             "z-score, heart rate only", "z-score, sleep only"]
    return pd.DataFrame(rows).set_index("Detector").loc[order].reset_index()


def score_all_nights(nights):
    """Final scores for every real night, as the app would show them.

    Two checks, heart rate and sleep, each with its own thresholds taken from clean
    nights: watch = top 2.5% (so about 5% of normal nights get a watch in total),
    alert = top 0.5%.
    """
    df = add_quality_flags(nights)
    dev = loo_deviations(df).join(df[["subject_id", "clean"]])
    spread = pooled_spread(dev, df["subject_id"].unique())
    z = z_scores(dev, spread)
    clean = dev["clean"]
    out = df.copy()
    thresholds = {}
    status = pd.Series("normal", index=df.index)
    rank = {"normal": 0, "watch": 1, "alert": 2}
    for name, feats in [("heart_rate", HR_FEATURES), ("sleep", SLEEP_FEATURES)]:
        score = z[feats].max(axis=1)
        watch, alert = np.nanquantile(score[clean], 0.975), np.nanquantile(score[clean], 0.995)
        thresholds[name] = {"watch": round(float(watch), 2), "alert": round(float(alert), 2)}
        out[f"{name}_score"] = score.round(2)
        s_ = np.where(score >= alert, "alert", np.where(score >= watch, "watch", "normal"))
        status = pd.Series([a if rank[a] >= rank[b] else b for a, b in zip(status, s_)], index=df.index)
    out["status"] = status.astype(str)
    out.loc[z.isna().any(axis=1), "status"] = "not enough history"
    out.loc[~out["clean"], "status"] = "data quality issue"
    out["reason"] = [explain(z.loc[i], df.loc[i], dev.loc[i]) if out.loc[i, "status"] in ("watch", "alert") else ""
                     for i in out.index]
    return out, spread, thresholds


nights = pd.read_csv("bidsleep_nights.csv")
flagged = add_quality_flags(nights)
print(f"{len(nights)} nights, {nights['subject_id'].nunique()} people, "
      f"{flagged['clean'].sum()} clean, {(~flagged['clean']).sum()} with a data-quality issue")
display(flagged[~flagged["clean"]][["subject_id", "night_folder", "hr_per_sleep_hour", "unknown_pct", "quality_issue"]])

## 2. How the personal baseline works

Each person has only 3-7 nights, too few to estimate their own night-to-night spread. So the baseline is split in two:

- **Centre:** the median of the same person's *other* clean nights (leave-one-night-out).
- **Spread:** the typical night-to-night variation, pooled across other people.
- **Score:** (tonight − centre) ÷ spread, counted only in the direction that matters (a higher heart rate, less sleep, more time awake).

In [ ]:
dev_all = loo_deviations(add_quality_flags(nights)).join(add_quality_flags(nights)[["subject_id", "clean"]])
spread_all = pooled_spread(dev_all, nights["subject_id"].unique())
print("Typical night-to-night change, pooled across 47 people:")
for f, v in spread_all.items():
    name, unit, _ = LABELS[f]
    print(f"  {name:<40} {v:5.2f} {'points' if unit.startswith('%') else unit}")

## 3. Test it with planted anomalies

There are no labelled illness nights, so we copy real clean nights and change them:

- heart rate raised by **+3, +5, +8 and +12 bpm**, or
- **disrupted sleep**: 1.5 h less sleep, +15 points of time awake, −8 points of deep sleep.

A planted night is never part of its own baseline. Testing is **5-fold, split by person**: thresholds and the pooled spread come from other people. Each threshold is set so that about 5% of untouched nights get flagged.

In [ ]:
res = evaluate(nights)
table = summary_table(res)
display(table)

# Detection by size of the heart-rate rise
x = [0] + SHIFTS_BPM
fig, ax = plt.subplots(figsize=(8, 4.5))
for _, row in table.iterrows():
    if row["Detector"] == "z-score, sleep only":
        continue
    y = [float(row["False alarms (untouched nights)"].rstrip("%"))] + [float(row[f"HR +{s} bpm"].rstrip("%")) for s in SHIFTS_BPM]
    ax.plot(x, y, marker="o", label=row["Detector"])
ax.set_xlabel("Planted rise in sleeping heart rate (bpm); 0 = untouched nights")
ax.set_ylabel("Nights flagged (%)")
ax.set_title("A rise of 8+ bpm is usually caught; 3 bpm is lost in normal variation")
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Reading the results:**

- **Heart rate alone is the most sensitive check for heart-rate changes.** Normal night-to-night variation is about ±4.5 bpm, so +3 bpm is hard to separate from noise, while +12 bpm is almost always caught.
- **Combining features has a cost.** One combined score must cover six chances of noise, so its threshold rises and heart-rate catches drop. Separate heart-rate and sleep checks keep more of that sensitivity. They pay for it with fewer disrupted-sleep catches, partly because the untouched nights already include some genuinely bad real nights.
- **On this real data, Isolation Forest does no better than the simple z-score.** It also can't explain itself as plainly. On our synthetic data the two looked similar, which is why testing on real data matters.

## 4. Score every real night (what the app would show)

The app uses two checks, heart rate and sleep. Each has thresholds from the clean nights: **watch** = its top 2.5%, **alert** = its top 0.5%.

In [ ]:
scored, spread, thresholds = score_all_nights(nights)
print("Thresholds:", thresholds)
print({str(k): int(v) for k, v in scored["status"].value_counts().items()})
cols = ["subject_id", "night_index", "rec_start_et", "status", "heart_rate_score", "sleep_score", "reason"]
display(scored[scored["status"].isin(["watch", "alert"])].sort_values(["status", "heart_rate_score"], ascending=False)[cols])

## 5. NASA Inspiration4: change from each crew member's own preflight levels

Each value is compared with that person's average over the 3 preflight draws (L-92, L-44, L-3). The spread is pooled across the 4 crew, since 3 draws are too few for a personal one. The race-based eGFR columns are dropped.

This is **descriptive only**: 4 people, a 3-day mission, general population lab ranges. We make no clinical claims.

In [ ]:
ORDER = ["L-92", "L-44", "L-3", "R+1", "R+45", "R+82", "R+194"]


def load_labs(path):
    d = pd.read_csv(path)
    d = d[~d["analyte"].str.startswith("eGFR")]  # race-based eGFR columns dropped
    return d.dropna(subset=["value"])


def personal_change(d):
    pre = d[d["phase"] == "preflight"]
    base = pre.groupby(["subject_id", "analyte"])["value"].agg(base_mean="mean", base_n="count").reset_index()
    pre = pre.merge(base, on=["subject_id", "analyte"])
    pre["resid"] = pre["value"] - pre["base_mean"]
    # pooled within-person preflight SD per analyte (degrees of freedom: n-1 per person)
    pooled = (pre.groupby("analyte")
                 .apply(lambda g: np.sqrt((g["resid"] ** 2).sum() /
                                          max((g.groupby("subject_id").size() - 1).sum(), 1)),
                        include_groups=False)
                 .rename("pooled_sd").reset_index())
    out = d.merge(base, on=["subject_id", "analyte"]).merge(pooled, on="analyte")
    out["change"] = out["value"] - out["base_mean"]
    out["z"] = out["change"] / out["pooled_sd"].replace(0, np.nan)
    out["outside_ref_range"] = (out["value"] < out["range_min"]) | (out["value"] > out["range_max"])
    out["timepoint"] = pd.Categorical(out["timepoint"], ORDER, ordered=True)
    return out


labs = personal_change(load_labs("i4_labs_with_baseline.csv"))
r1 = labs[labs["timepoint"] == "R+1"]
moves = (r1.groupby("analyte")
           .agg(mean_change=("change", "mean"), mean_z=("z", "mean"),
                crew_below_baseline=("z", lambda s: int((s < -2).sum())),
                crew_above_baseline=("z", lambda s: int((s > 2).sum())),
                outside_lab_range=("outside_ref_range", "sum"), units=("units", "first"))
           .assign(abs_z=lambda t: t["mean_z"].abs()).sort_values("abs_z", ascending=False).drop(columns="abs_z"))
print("Day after landing (R+1): biggest changes from each crew member's own preflight average")
display(moves.head(8).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, analyte in zip(axes, ["MCV", "SODIUM"]):
    g = labs[labs["analyte"] == analyte]
    for cid, s in g.groupby("subject_id"):
        s = s.sort_values("timepoint")
        ax.plot(s["timepoint"].astype(str), s["value"], marker="o", label=cid)
    lo, hi, unit = g["range_min"].iloc[0], g["range_max"].iloc[0], g["units"].iloc[0]
    ax.set_title(f"{ {'MCV': 'MCV', 'SODIUM': 'Sodium'}[analyte]} ({unit}); lab range {lo:g}-{hi:g}")
    ax.axvline(2.5, color="grey", ls="--", lw=1)
    ax.text(2.55, ax.get_ylim()[1], " flight", va="top", fontsize=8, color="grey")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.suptitle("Every crew member's MCV dropped after flight, yet stayed inside the normal lab range", fontsize=11)
plt.tight_layout()
plt.show()

This is the app's whole argument in real astronaut data. All four crew members' MCV (average red blood cell size) was lower the day after landing than their own preflight average, but **every value stayed inside the normal lab range**. A range check would show green; a personal baseline shows the change.

This notebook doesn't explain why it happened. That is a question for the flight surgeon, not the app.

## 6. Export for the app

In [ ]:
export = {
    "source": "PhysioNet bidsleep v1.0.1 (general human data, Earth; not astronauts)",
    "method": "Leave-one-night-out personal median; night-to-night spread pooled across people; separate heart-rate and sleep checks",
    "thresholds": thresholds,
    "pooled_spread": {k: round(float(v), 3) for k, v in spread.items()},
    "nights": json.loads(scored[["subject_id", "night_index", "rec_start_et", "status", "heart_rate_score",
                                 "sleep_score", "reason", "quality_issue"]].astype({"status": str}).to_json(orient="records")),
}
Path("nights_scored.json").write_text(json.dumps(export, indent=1))
labs.assign(timepoint=labs["timepoint"].astype(str))[
    ["subject_id", "sex", "analyte", "units", "timepoint", "day", "phase", "value", "range_min", "range_max",
     "base_mean", "pooled_sd", "change", "z", "outside_ref_range"]
].to_csv("i4_personal_changes.csv", index=False)
print("Saved nights_scored.json and i4_personal_changes.csv")
try:
    from google.colab import files
    files.download("nights_scored.json")
    files.download("i4_personal_changes.csv")
except ImportError:
    pass

## Limitations

- **PhysioNet subjects are healthy adults on Earth**, not astronauts. The model shows the method; it is not validated on crew.
- **Planted anomalies are our assumptions** about what a change looks like. No real illness labels exist.
- **Inspiration4 is 4 people on a 3-day flight**, so it is descriptive only. Lab ranges are general population ranges, not astronaut medical limits.
- **The output is an anomaly score with a reason**, not a diagnosis or treatment advice.

**Citations:** PhysioNet bidsleep v1.0.1 (ODC-By 1.0). NASA OSDR OSD-569 and OSD-575.

**AI use:** drafted with Claude (Anthropic), reviewed and run by the team.